<h3>한국어 임베딩과 캐시</h3>
<p>문장 하나의 벡터를 먼저 관찰하고, 여러 문장을 한 번에 변환합니다. 마지막에는 LangChain의 캐시 기능으로 첫 요청의 벡터를 파일에 저장하고, 같은 문서를 다시 요청할 때 그 벡터를 읽습니다.</p>

In [1]:
# OpenAI 임베딩과 LangChain의 파일 캐시 구현을 설치합니다.
%pip install -q "python-dotenv==1.2.3" "langchain-openai==1.6.0"     "langchain-classic==1.0.8"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings

# 현재 노트북 폴더부터 상위 폴더에서 키 파일을 찾습니다.
import os

env_file = find_dotenv(usecwd=True)
assert env_file, "실습 최상위 폴더에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."

# dimensions=256은 교육용 실험에서 출력과 저장 크기를 줄이기 위한 설정입니다.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=256,
)

<h3>1단계: 문장 하나를 벡터로 바꾸기</h3>
<p>벡터 전체를 출력하면 숫자가 너무 많습니다. 앞의 다섯 값과 전체 길이만 확인합니다.</p>

In [3]:
sentence = "제주 배송은 기본 배송보다 이틀 더 걸립니다."

# embed_documents()는 문자열 리스트를 받고, 각 문자열에 대응하는 벡터를 반환합니다.
single_vector = embeddings.embed_documents([sentence])[0]

print("앞의 다섯 값:", single_vector[:5])
print("벡터 길이:", len(single_vector))

앞의 다섯 값: [0.00563812255859375, 0.01345062255859375, 0.080322265625, 0.038909912109375, -0.028472900390625]
벡터 길이: 256


<h3>2단계: 여러 문장을 한 번에 변환하기</h3>
<p>입력 문장 수와 반환된 벡터 수가 같은지 확인합니다. 모든 벡터는 같은 임베딩 설정을 사용하므로 길이도 같습니다.</p>

In [4]:
# 문장 목록을 한 번에 변환하고 입력 개수와 벡터 개수를 비교합니다.
sentences = [
    "제주 배송은 기본 배송보다 이틀 더 걸립니다.",
    "환불은 수령 후 7일 이내 신청합니다.",
    "회원 등급은 매월 갱신합니다.",
]
vectors = embeddings.embed_documents(sentences)

print("문장 수:", len(sentences))
print("벡터 수:", len(vectors))
print("각 벡터 길이:", [len(vector) for vector in vectors])

문장 수: 3
벡터 수: 3
각 벡터 길이: [256, 256, 256]


<h3>3단계: 차원 하나 바꾸어 벡터 길이 비교</h3><p>문장과 모델은 유지하고 dimensions만 256에서 128로 바꿉니다. 반환 벡터의 길이를 비교합니다.</p>

In [5]:
# 같은 문장을 같은 모델에 보내고 차원 설정만 바꿉니다.
short_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", dimensions=128,
)
short_vector = short_embeddings.embed_documents([sentence])[0]
print("기준 벡터 길이:", len(single_vector))
print("변경 벡터 길이:", len(short_vector))
assert len(short_vector) == 128

기준 벡터 길이: 256
변경 벡터 길이: 128


<h3>4단계: LangChain 캐시 사용하기</h3>
<p>이번 커널에서만 사용하는 빈 임시 폴더를 만듭니다. 같은 문서 목록을 두 번 요청합니다. 첫 요청 뒤 생긴 벡터 파일 수가 두 번째 요청 뒤에도 그대로인지 확인합니다.</p>

In [6]:
from pathlib import Path
from tempfile import TemporaryDirectory

from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

# TemporaryDirectory는 매 실행마다 비어 있는 새 폴더를 제공합니다.
cache_directory = TemporaryDirectory(prefix="ch10-embedding-cache-")
# LocalFileStore는 이 폴더에 임베딩 결과를 파일로 보관합니다.
byte_store = LocalFileStore(cache_directory.name)
# 원래 임베딩 모델에 파일 저장소를 연결하면 같은 문서의 벡터를 재사용합니다.
# namespace는 모델·차원 설정을 구분하고 sha256은 텍스트를 저장용 키로 변환합니다.
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    byte_store,
    namespace="text-embedding-3-small-256-",
    key_encoder="sha256",
)

In [7]:
cache_path = Path(cache_directory.name)
files_before = list(cache_path.rglob("*"))

# 첫 요청 뒤에는 문서 임베딩을 저장한 캐시 파일이 생깁니다.
first_vectors = cached_embeddings.embed_documents(sentences)
files_after_first = [path for path in cache_path.rglob("*") if path.is_file()]

# 같은 입력을 다시 요청하면 기존 파일을 사용하므로 파일 수가 유지됩니다.
second_vectors = cached_embeddings.embed_documents(sentences)
files_after_second = [path for path in cache_path.rglob("*") if path.is_file()]

print("실행 전 파일 수:", len(files_before))
print("첫 요청 뒤 파일 수:", len(files_after_first))
print("두 번째 요청 뒤 파일 수:", len(files_after_second))
print("두 결과가 같습니다:", first_vectors == second_vectors)

실행 전 파일 수: 0
첫 요청 뒤 파일 수: 3
두 번째 요청 뒤 파일 수: 3
두 결과가 같습니다: True


<h3>일부러 실패해 보기</h3>
<p>서로 다른 길이의 벡터는 같은 벡터 공간에서 비교할 수 없습니다. 저장 전에 차원을 검사합니다.</p>

In [8]:
try:
    # 두 번째 벡터만 잘라 차원이 다른 상태를 의도적으로 만듭니다.
    mixed_vectors = [vectors[0], vectors[1][:128]]
    dimensions = {len(vector) for vector in mixed_vectors}
    assert len(dimensions) == 1, "벡터 차원이 서로 다릅니다"
except AssertionError as error:
    print(type(error).__name__, str(error))
    print("발견한 차원:", sorted(dimensions))

AssertionError 벡터 차원이 서로 다릅니다
발견한 차원: [128, 256]


<p><strong>수정 방법</strong> 같은 모델과 차원으로 만든 원본 벡터만 사용하고 차원이 256인지 확인합니다.</p>

In [9]:
# 잘라 내기 전 원본 벡터들로 같은 차원인지 다시 확인합니다.
consistent_vectors = vectors
dimensions = {len(vector) for vector in consistent_vectors}

assert dimensions == {256}
print("벡터 수:", len(consistent_vectors))
print("확인한 차원:", sorted(dimensions))

벡터 수: 3
확인한 차원: [256]
